# BNMP — Extração bronze (API do BNMP 2.0)

Coleta os dados do Banco Nacional de Monitoramento de Prisões (BNMP 2.0,
PDPJ/CNJ) e grava os JSON brutos no Lakehouse `mp_bronze`, seção Files, sob
`bnmp/json/`, usando `python/src/modulos/bnmp/` do repositório
`mpsp-jurimetria/proj202607`.

O que é gravado:
- `bnmp/json/contexto_sessao.json`, `dominios.json`, `status_pessoas.json`
- `bnmp/json/{recurso}/{consulta}/pagina_NNNNNN.json` — uma página por arquivo
- `bnmp/json/{recurso}/{consulta}/_manifesto.json` — progresso da coleta

**Pré-requisitos:**
- Segredos no Key Vault `KV-Jurimetria`: `BNMP-USUARIO`, `BNMP-SENHA` e
  `BNMP-OTP-SECRET` (segredo base32 do app autenticador).
- O 2º fator já deve estar **cadastrado** na conta do BNMP. Se o SSO ainda
  pedir o cadastro (required action `CONFIGURE_TOTP`), conclua uma vez pelo
  navegador guardando o segredo base32 — enquanto isso não for feito, o
  Keycloak gera um segredo novo a cada login e a automação falha com essa
  mensagem.
- Identidade do notebook com leitura no Key Vault e escrita no `mp_bronze`.

A coleta é **retomável**: reexecutar a célula de execução pula as páginas já
gravadas no Lakehouse. Coletas longas (milhões de registros) sobrevivem à
expiração da sessão do SSO (~8h) — o cliente refaz o login sozinho.

In [ ]:
%pip install --quiet git+https://github.com/mpsp-jurimetria/proj202607.git#subdirectory=python

## Configuração

Os IDs do Fabric não são segredos, mas evite deixar valores reais commitados
aqui. As credenciais do BNMP vêm sempre do Key Vault.

In [ ]:
import os

from notebookutils import credentials

KV_URI = "https://KV-Jurimetria.vault.azure.net"
os.environ["BNMP_USER"] = credentials.getSecret(KV_URI, "BNMP-USUARIO")
os.environ["BNMP_PASSWORD"] = credentials.getSecret(KV_URI, "BNMP-SENHA")
os.environ["BNMP_OTP_SECRET"] = credentials.getSecret(KV_URI, "BNMP-OTP-SECRET")

# 39 = Ministério Público do Estado de São Paulo
os.environ["BNMP_ORGAO_ATIVO"] = "39"

os.environ["FABRIC_WORKSPACE_ID"] = "<id do workspace>"
os.environ["FABRIC_LAKEHOUSE_ID"] = "<id do lakehouse mp_bronze>"

## Execução

A API recusa paginação além do registro 10.000, então uma consulta ampla (uma
UF inteira tem centenas de milhares de pessoas) **não** pode ser coletada por
inteiro. Passando `uf_ids_pessoas`, o planejador mede cada consulta e a
subdivide por status, sexo e município até cada parte caber no limite; o plano
resultante fica em `bnmp/json/pessoas/_plano.json`.

Confira nesse plano o campo `registros_inalcancaveis`: é quanto ficaria de fora
por partições que continuam acima do teto mesmo após a subdivisão.

O planejamento consome uma requisição por sondagem, então a primeira execução
de uma UF grande leva alguns minutos antes de começar a baixar páginas.

In [ ]:
from src.modulos.bnmp.etl.extract_bronze import executar

# 26 = São Paulo; acrescente outras UFs para ampliar a coleta
executar(uf_ids_pessoas=[26], tamanho_pagina=1000)

## Verificação

In [ ]:
import json

from src.infra.lakehouse import download_bytes
from src.modulos.bnmp.etl import read_bronze

dominios = read_bronze.ler_dominios()
print(f"dominios: {len(dominios)} listas de referência")

plano = json.loads(download_bytes("bnmp/json/pessoas/_plano.json"))
print(f"partições: {plano['particoes']}")
print(f"registros previstos: {plano['registros_previstos']}")
print(f"acima do limite: {plano['particoes_acima_do_limite']} partições, "
      f"{plano['registros_inalcancaveis']} registros fora do alcance")

# manifesto da maior partição coletada
maior = max(plano["particoes_detalhe"], key=lambda p: p["total"])
manifesto = read_bronze.ler_manifesto("pessoas", maior["rotulo"])
print(maior["rotulo"], "->", manifesto["status"],
      manifesto["paginas_gravadas"], "de", manifesto["total_paginas"], "páginas")

envelope = read_bronze.ler_pagina("pessoas", maior["rotulo"], 0)
print("1º registro:", envelope["resposta"]["content"][0]["dadosGeraisPessoa"]["nome"])